#Dataset Assessment and Initial Exploratory Analysis

**Author:** Stella Dong

*Department of Computer Science & Engineering, Texas A&M University*  
*This work was completed as part of CSCE 676: Data Mining and Analysis, taught by Prof. James Caverlee.*

The Github repository of this project can be found here: [Github link](https://github.com/stellasdong/676_project/).

---


In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
path = "/content/drive/MyDrive/TAMU/data_mining/project_data"
os.chdir(path)

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

## Identification of Candidate Datasets

### Dataset 1: [Texas Inpatient PUF](https://www.dshs.texas.gov/center-health-statistics/texas-health-care-information-collection/download-and-purchase-data/texas-inpatient-public-use-data-file-pudf/public-use-data-File-pudf-inpatient-free-download)

This is a publicly available dataset of inpatient data across around 700 hospitals in Texas. Although it is completely anonymized, the dataset still contains hospital data, discharge data, diagnoses, stay durations, etc., that makes this dataset very rich. Course topics that this dataset align with include frequent itemsets and clustering, and beyond the course there is potential for sequence analysis or probabilistic record linkage. This dataset covers four quarters of hospital data, with 167 variables for around 800,000 records per quarter. There is a variety of data types for the variables, including address data, record IDs, temporal data, billing and charge data, and diagnosis codes. My target variable would likely be principal diagnosis code. Because this is anonymized health data, I am not permitted to use this dataset to link to other personally identifiable records or in anyway attempt to identify patients or physicians.


### Dataset 2: [NC Voter Registry Snapshot - Orange County](https://www.ncsbe.gov/results-data/voter-registration-data)

This is a subset of the North Carolina Voter Registration Data, isolated to just Orange County. Course topics that this data could align with is clustering to link households and large-scale ML to predict party affiliation. Beyond course techniques, this dataset could be used for temporal analysis, using the voter registration across multiple years to analyze shifts in party affiliation. The dataset includes a lot of variables, mostly strings, of addresses, personal information like name, gender and age, anonymized contact information, and their registered party code, which would be my target variable.
There are no limits to the usage of this dataset.

### Dataset 3: [CMS Health Insurance Exchange Rate PUF](https://download.cms.gov/marketplace-puf/2026/rate-puf.zip)

This is a dataset released by the CMS containing plan-level data on rates based on different factors for an eligible subscriber, like tobacco use, age and location. Course topics that this dataset could align with is large-scale ML to train a regression model to predict individual rate, clustering by region to find hidden high-cost zones, anomaly detection and streams. Beyond course techniques, this dataset lends itself well to surcharge strategy analysis to calculate any tobacco premium on insurance rates. The dataset has 25 variables for over 2 million rates. There's a large variety of data types, including IDs for the issuer and the plan, the rate, geographic data for where the plan is available and qualifying data on the subscriber, like their age and tobacco usage. The target variable for this dataset would be the rate. There are no limits to the usage of this dataset.

## Comparative Analysis of Datasets

| Dataset | Supported Tasks | Data Qual Issues | Alg. Feasibility | Bias | Ethics |
| :--- | :--- | :--- |  :--- | :--- | :--- |
| Texas Inpatient | frequent itemsets (course), <br>sequence analysis (external) | lagging in reporting causing repeats across quarters, missing values and invalid codes, anonymization causes a lot of noise | large amount of data across all of Texas for even just one quarter of 2019 | data reflects who accessed care, not who needed it, so findings can only be applied to those with healthcare access | risk of reidentification by performing analysis on this data that could expose patients |
| NC VR | clustering (course),<br> large-scale ML (course), <br> temporal analysis (external) | high volume of missing data for variables, including target variable: registered party code | Orange County is small enough for most ML algorithms without Spark, though expanding to other counties would need it | only one county is not representative of even the state, so biased towards Orange County, which is demographically different from surrounding counties due to UNC | analyzing party shifts at a household level can feel invasive |
| CMS Rates | large-scale ML (course), <br> surcharge strategy analysis (external) | missing variable data, mapping requires external datasets | a simple regression would be feasible on millions of rows, but more complex ML would need more memory | plans exclude states with their own platforms, which would cause bias in the national average | surcharge analysis could unintentionally highlight ways for issuers to maximize costs for vulnerable groups |



In [3]:
# NC Voter Registry
nc_voter = pd.read_csv("ncvoter68.txt", sep='\t',encoding='latin1')
nc_voter.info()

/tmp/ipython-input-2129948057.py:1: DtypeWarning: Columns (18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  nc_voter = pd.read_csv("ncvoter68.txt", sep='\t',encoding='latin1')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140524 entries, 0 to 140523
Data columns (total 70 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   county_id                 140524 non-null  int64  
 1   county_desc               140524 non-null  object 
 2   voter_reg_num             140524 non-null  int64  
 3   ncid                      140524 non-null  object 
 4   last_name                 140517 non-null  object 
 5   first_name                140520 non-null  object 
 6   middle_name               126052 non-null  object 
 7   name_suffix_lbl           4783 non-null    object 
 8   status_cd                 140524 non-null  object 
 9   voter_status_desc         140524 non-null  object 
 10  reason_cd                 140524 non-null  object 
 11  voter_status_reason_desc  140524 non-null  object 
 12  res_street_address        140524 non-null  object 
 13  res_city_desc             114959 non-null  o

In [4]:
# CMS RATES PUF
rate_puf = pd.read_csv("rate-puf.csv")
rate_puf.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2235761 entries, 0 to 2235760
Data columns (total 20 columns):
 #   Column                                     Dtype  
---  ------                                     -----  
 0   BusinessYear                               int64  
 1   StateCode                                  object 
 2   IssuerId                                   int64  
 3   SourceName                                 object 
 4   ImportDate                                 object 
 5   RateEffectiveDate                          object 
 6   RateExpirationDate                         object 
 7   PlanId                                     object 
 8   RatingAreaId                               object 
 9   Tobacco                                    object 
 10  Age                                        object 
 11  IndividualRate                             float64
 12  IndividualTobaccoRate                      float64
 13  Couple                                    

In [10]:
# TEXAS INPATIENT DATA
tx_inpatient_q1 = pd.read_csv("pudf/PUDF_base1_1q2019_tab.txt", sep="\t")
tx_inpatient_q2 = pd.read_csv("pudf/PUDF_base1_2q2019_tab.txt", sep="\t")
tx_inpatient_q3 = pd.read_csv("pudf/PUDF_base1_3q2019_tab.txt", sep="\t")
tx_inpatient_q4 = pd.read_csv("pudf/PUDF_base1_4q2019_tab.txt", sep="\t")

tx_inpatient = pd.DataFrame(np.concatenate([
    tx_inpatient_q1.values,
    tx_inpatient_q2.values,
    tx_inpatient_q3.values,
    tx_inpatient_q4.values]),
    columns=tx_inpatient_q1.columns
)
tx_inpatient.info()

/tmp/ipython-input-3980787014.py:2: DtypeWarning: Columns (4,6,7,11,15,17,18,21,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,103,105,107,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,128,130,132,134,136,138,140,142,144,146,148,150) have mixed types. Specify dtype option on import or set low_memory=False.
  tx_inpatient_q1 = pd.read_csv("pudf/PUDF_base1_1q2019_tab.txt", sep="\t")
/tmp/ipython-input-3980787014.py:3: DtypeWarning: Columns (4,6,7,11,15,17,18,21,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,103,105,107,122,124,126,128,130,132,134,136,138,140,142,144,146,148,150) have mixed types. Specify dtype option on import or set low_memory=False.
  tx_inpatient_q2 = pd.read_csv("pudf/PUDF_base1_2q2019_tab.txt", sep="\t")
/tmp/ipython-input-3980787014.py:4: DtypeWarning: Columns (4,6,7,11,15,17,18,21,68,69,70,71,72,73,74,75,76,77,78,79,80,81,87,88,89,90,91,92,93,94

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3141275 entries, 0 to 3141274
Columns: 168 entries, RECORD_ID to Unnamed: 167
dtypes: object(168)
memory usage: 3.9+ GB


In [11]:
tx_inpatient.head()

,RECORD_ID,DISCHARGE,THCIC_ID,TYPE_OF_ADMISSION,SOURCE_OF_ADMISSION,SPEC_UNIT_1,SPEC_UNIT_2,SPEC_UNIT_3,SPEC_UNIT_4,SPEC_UNIT_5,...,RISK_MORTALITY,ILLNESS_SEVERITY,APR_GROUPER_VERSION_NBR,APR_GROUPER_ERROR_CODE,ATTENDING_PHYSICIAN_UNIF_ID,OPERATING_PHYSICIAN_UNIF_ID,ENCOUNTER_INDICATOR,PROVIDER_NAME,EMERGENCY_DEPT_FLAG,Unnamed: 167
0,120190000375,2019Q1,703003,3.0,2,I,NaN,NaN,NaN,NaN,...,2,2,7360,0,9999999998,9999999998.0,1,Corpus Christi Medical Center-Heart Hospital,N,NaN
1,120190000376,2019Q1,703003,1.0,1,I,NaN,NaN,NaN,NaN,...,4,4,7360,0,1902929264,9999999998.0,1,Corpus Christi Medical Center-Heart Hospital,Y,NaN
2,120190000377,2019Q1,703003,3.0,2,I,NaN,NaN,NaN,NaN,...,2,2,7360,0,9999999998,9999999998.0,1,Corpus Christi Medical Center-Heart Hospital,N,NaN
3,120190000378,2019Q1,703003,3.0,2,NaN,NaN,NaN,NaN,NaN,...,1,1,7360,0,9999999998,9999999998.0,1,Corpus Christi Medical Center-Heart Hospital,N,NaN
4,120190000379,2019Q1,703003,3.0,2,I,NaN,NaN,NaN,NaN,...,1,2,7360,0,9999999998,9999999998.0,1,Corpus Christi Medical Center-Heart Hospital,N,NaN


In [17]:
# TEXAS INPATIENT DATA exclusive to Bryan-College Station facilities
BCS_THCIC_ID = ["975162", "975270", "717500", "002001", "206100", "975403", "976329"]
tx_inpatient['THCIC_ID'] = tx_inpatient['THCIC_ID'].astype(str)
bcs_tx_inpatient = tx_inpatient[tx_inpatient['THCIC_ID'].isin(BCS_THCIC_ID)]
bcs_tx_inpatient.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11174 entries, 34302 to 3040051
Columns: 168 entries, RECORD_ID to Unnamed: 167
dtypes: object(168)
memory usage: 14.4+ MB


## Dataset Selection

The dataset that I eventually decided on is the Texas Inpatient Dataset. I found that this dataset was the richest in variety of variables, so there are more opportunities to explore different analyses. Primarily, I chose this dataset because it supports finding frequent itemsets for common groups of diagnoses in patients. It also supports clustering for similar patients, then large-scale ML to predict a variety of variables, like length of stay, total charges, etc. External from coursework, this would support sequential analysis, to predict returns to hospitals based on features like diagnoses.

The trade-offs for this dataset is that this dataset requires linking to other datasets for important information about the facilities and the costs. There is also limited opportunities for any graph representation because there is no defined relationship between entries. It is also a very large dataset, but I was able to circumvent this problem by first scaling the dataset down from the entire state (4GB) to just Bryan-College Station facilities (14.4 MB), and if time permits, I will scale my analyses up to a larger area.

## Exploratory Data Analysis

## Initial Insights and Direction

## Collaboration Declaration
On my honor, I declare the following resources:
1. Collaborators:
-

2. Web Sources:
-

3. AI Tools:
-

4. Citations:
- Texas Hospital Inpatient Discharge Public Use Data File, 2019. Texas Department of State Health Services, Center for Health
Statistics, Austin, Texas. https://www.dshs.texas.gov/center-health-statistics/texas-health-care-information-collection/download-and-purchase-data/texas-inpatient-public-use-data-file-pudf/public-use-data-File-pudf-inpatient-free-download
- Centers for Medicare & Medicaid Services. (2025). 2026 Health Insurance Exchange Public Use Files
(Rates PUF). Retrieved from
http://www.cms.gov/CCIIO/Resources/Data-Resources/marketplace-puf.html
- NC Voter Registry: https://www.ncsbe.gov/results-data/voter-history-data
- Reporting Status of Texas Hospitals, 2024: https://www.dshs.texas.gov/sites/default/files/thcic/hospitals/inpatientstatusreport4q2024.pdf